In [16]:
# IMPORTS

import os
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

In [17]:
# CHEMINS

base_path = "../data/raw/mri_dataset_brain_cancer_oc/"

cancer_path = os.path.join(base_path, "avec_labels/cancer")
normal_path = os.path.join(base_path, "avec_labels/normal")
unlabeled_path = os.path.join(base_path, "sans_label")

# Vérification
assert os.path.exists(cancer_path), "Dataset introuvable, voir README"

In [18]:
# PREPROCESSING

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [19]:
# MODÈLE RESNET

model = models.resnet50(pretrained=True)

# Suppression de la couche finale
model = torch.nn.Sequential(*list(model.children())[:-1])

# Gel des poids
for param in model.parameters():
    param.requires_grad = False

# Mode évaluation
model.eval()

c:\Users\selma\Desktop\BrainScanAI\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\selma\Desktop\BrainScanAI\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [20]:
# FONCTION D'EXTRACTION

def extract_features(image_path):
    try:
        img = Image.open(image_path).convert("RGB")
        img = transform(img)
        img = img.unsqueeze(0)

        with torch.no_grad():
            features = model(img)

        return features.squeeze().numpy()

    except Exception as e:
        print(f"[ERREUR] {image_path} -> {e}")
        return None

In [ ]:
# EXTRACTION DES FEATURES

features = []
labels = []
paths = []

# cancer
for f in tqdm(os.listdir(cancer_path)):
    path = os.path.join(cancer_path, f)
    feat = extract_features(path)

    if feat is not None:
        features.append(feat)
        labels.append("cancer")
        paths.append(os.path.relpath(path, start=base_path).replace("\\", "/"))
        
# normal
for f in tqdm(os.listdir(normal_path)):
    path = os.path.join(normal_path, f)
    feat = extract_features(path)

    if feat is not None:
        features.append(feat)
        labels.append("normal")
        paths.append(os.path.relpath(path, start=base_path))

# unlabeled
for f in tqdm(os.listdir(unlabeled_path)):
    path = os.path.join(unlabeled_path, f)
    feat = extract_features(path)

    if feat is not None:
        features.append(feat)
        labels.append("unlabeled")
        paths.append(os.path.relpath(path, start=base_path))

print("Images traitées :", len(features))

100%|██████████| 1406/1406 [02:14<00:00, 10.42it/s]

Images traitées : 1506


In [22]:
# DATAFRAME

X = np.array(features)

df = pd.DataFrame(X)
df["label"] = labels
df["path"] = paths

df.head()

,0,1,2,3,4,5,6,7,8,9,...,2040,2041,2042,2043,2044,2045,2046,2047,label,path
0,0.409749,0.873298,0.174183,0.301675,0.067051,0.273842,0.247952,0.647732,0.959357,0.151482,...,0.563092,0.128302,0.720562,0.032226,0.519088,0.430458,1.099062,0.938813,cancer,avec_labels\cancer\05340cd4-3bb2-459d-9937-bf2...
1,0.650230,0.721211,0.054125,0.181203,0.000000,0.292601,0.964571,0.697341,0.577685,0.305469,...,0.185135,0.182850,0.192730,0.043186,0.241807,0.485734,0.671984,0.706141,cancer,avec_labels\cancer\0c6f3641-60d9-4a76-abe5-de8...
2,1.192965,1.131451,0.107712,0.199649,0.000219,0.451692,0.180534,0.693340,0.275049,0.544788,...,0.057330,0.250783,0.273103,0.090985,0.080423,0.438565,0.297450,1.483259,cancer,avec_labels\cancer\0f718241-8f63-4b55-81ce-315...
3,0.494694,0.456065,0.174034,0.257347,0.115510,0.877980,0.727288,0.369569,0.536643,0.290675,...,0.120582,0.462077,0.182437,0.096539,0.537069,0.563836,0.950194,1.112594,cancer,avec_labels\cancer\11a7a426-4806-401e-98b2-b96...
4,0.491905,0.354549,0.390202,0.040721,0.014974,0.305518,0.158039,0.126338,0.265495,0.287022,...,0.830522,0.075254,0.130555,0.143496,0.158412,0.429747,1.463409,1.068511,cancer,avec_labels\cancer\1c043dbb-4623-4769-8e5e-022...


In [23]:
# SAUVEGARDE

df.to_csv("../outputs/features.csv", index=False)

## Extraction de features

Les images ont été prétraitées (redimensionnement, normalisation) afin d’être compatibles avec le modèle ResNet50.

Le modèle pré-entraîné a été utilisé comme extracteur de caractéristiques :
- suppression de la couche de classification
- conservation des couches convolutionnelles
- gel des poids

Chaque image est transformée en un vecteur de dimension 2048.

Ces vecteurs sont stockés dans un DataFrame et seront utilisés pour :
- le clustering
- l’apprentissage semi-supervisé